# Feature Construction and Feature Splitting

## Feature Construction
Creating **new columns** by combining or transforming existing ones.
```
SibSp + Parch + 1 → Family_size   (new meaningful column)
```
The model can learn better from the new column than from the raw ones.

## Feature Splitting
**Breaking one column** into multiple smaller, more useful columns.
```
Name: 'Braund, Mr. Owen Harris'
→ Title: 'Mr'   (extracted from Name)
```
A single column often contains hidden information that the model can't use directly.

In [5]:
# Import required libraries
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

In [6]:
# Load only relevant columns from Titanic dataset
df = pd.read_csv('train.csv')[['Age', 'Pclass', 'SibSp', 'Parch', 'Survived']]

In [7]:
# Preview first 5 rows
df.head()

,Age,Pclass,SibSp,Parch,Survived
0,22.0,3,1,0,0
1,38.0,1,1,0,1
2,26.0,3,0,0,1
3,35.0,1,1,0,1
4,35.0,3,0,0,0


In [8]:
# Drop rows with missing values — Age has NaN values
# We need complete data to train the model
df.dropna(inplace=True)
df.head()

,Age,Pclass,SibSp,Parch,Survived
0,22.0,3,1,0,0
1,38.0,1,1,0,1
2,26.0,3,0,0,1
3,35.0,1,1,0,1
4,35.0,3,0,0,0


In [9]:
# Separate features and target
X = df.iloc[:, 0:4]   # Age, Pclass, SibSp, Parch
y = df.iloc[:, -1]    # Survived

X.head()

,Age,Pclass,SibSp,Parch
0,22.0,3,1,0
1,38.0,1,1,0
2,26.0,3,0,0
3,35.0,1,1,0
4,35.0,3,0,0


In [10]:
# Baseline accuracy BEFORE feature construction
# cross_val_score trains and tests the model 20 times on different splits
# np.mean gives the average accuracy across all 20 runs
baseline_score = np.mean(cross_val_score(LogisticRegression(), X, y, scoring='accuracy', cv=20))
print(f"Accuracy before feature construction: {baseline_score * 100:.2f}%")

Accuracy before feature construction: 69.33%


## Part 1 — Feature Construction

In [11]:
# Create new column: Family_size
# SibSp = number of siblings/spouses aboard
# Parch = number of parents/children aboard
# +1 = the passenger themselves
# Family_size tells us whether the passenger was alone or with family
X = X.copy()
X['Family_size'] = X['SibSp'] + X['Parch'] + 1

X.head()

,Age,Pclass,SibSp,Parch,Family_size
0,22.0,3,1,0,2
1,38.0,1,1,0,2
2,26.0,3,0,0,1
3,35.0,1,1,0,2
4,35.0,3,0,0,1


In [12]:
# Create a function to categorize family size into 3 groups
# This makes the feature even more meaningful for the model
def family_type(num):
    if num == 1:
        return 0   # alone
    elif num > 1 and num <= 4:
        return 1   # small family
    else:
        return 2   # large family

# Test the function
print(family_type(1))  # alone
print(family_type(3))  # small family
print(family_type(6))  # large family

0
1
2


In [13]:
# Apply the function to every row using .apply()
# .apply() runs the function on each value in the column
X['Family_type'] = X['Family_size'].apply(family_type)

X.head()

,Age,Pclass,SibSp,Parch,Family_size,Family_type
0,22.0,3,1,0,2,1
1,38.0,1,1,0,2,1
2,26.0,3,0,0,1,0
3,35.0,1,1,0,2,1
4,35.0,3,0,0,1,0


In [14]:
# Drop the original columns that are now captured in Family_type
# SibSp, Parch and Family_size are no longer needed
# Family_type alone summarizes all that information
X.drop(columns=['SibSp', 'Parch', 'Family_size'], inplace=True)

X.head()

,Age,Pclass,Family_type
0,22.0,3,1
1,38.0,1,1
2,26.0,3,0
3,35.0,1,1
4,35.0,3,0


In [15]:
# Accuracy AFTER feature construction
# Did adding Family_type improve the model?
new_score = np.mean(cross_val_score(LogisticRegression(), X, y, scoring='accuracy', cv=20))
print(f"Accuracy before: {baseline_score * 100:.2f}%")
print(f"Accuracy after:  {new_score * 100:.2f}%")
print(f"Improvement:     {(new_score - baseline_score) * 100:.2f}%")

Accuracy before: 69.33%
Accuracy after:  70.03%
Improvement:     0.70%


## Part 2 — Feature Splitting

In [16]:
# Reload full dataset — we need the Name column now
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [17]:
# Look at the Name column
# Format: 'LastName, Title. FirstName MiddleName'
# Example: 'Braund, Mr. Owen Harris'
# We can extract the Title (Mr, Mrs, Miss, Dr etc.) which is useful info
df['Name']

0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina
3           Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                               Allen, Mr. William Henry
                             ...                        
886                                Montvila, Rev. Juozas
887                         Graham, Miss. Margaret Edith
888             Johnston, Miss. Catherine Helen "Carrie"
889                                Behr, Mr. Karl Howell
890                                  Dooley, Mr. Patrick
Name: Name, Length: 891, dtype: str

In [18]:
# Extract Title from Name using string splitting
# Step 1: split by ', ' → get everything after the comma → 'Mr. Owen Harris'
# Step 2: split by '.' → get everything before the dot → 'Mr'
df['Title'] = df['Name'].str.split(', ', expand=True)[1].str.split('.', expand=True)[0]

df[['Name', 'Title']].head(10)

,Name,Title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,"Allen, Mr. William Henry",Mr
5,"Moran, Mr. James",Mr
6,"McCarthy, Mr. Timothy J",Mr
7,"Palsson, Master. Gosta Leonard",Master
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",Mrs
9,"Nasser, Mrs. Nicholas (Adele Achem)",Mrs


In [19]:
# Check survival rate by Title — sorted highest to lowest
# This shows how useful Title is as a feature
# Mrs and Miss (females) have much higher survival rate than Mr (males)
(df.groupby('Title').mean(numeric_only=True)['Survived']).sort_values(ascending=False)

Title
the Countess    1.000000
Mlle            1.000000
Sir             1.000000
Ms              1.000000
Lady            1.000000
Mme             1.000000
Mrs             0.792000
Miss            0.697802
Master          0.575000
Col             0.500000
Major           0.500000
Dr              0.428571
Mr              0.156673
Jonkheer        0.000000
Rev             0.000000
Don             0.000000
Capt            0.000000
Name: Survived, dtype: float64

In [20]:
# Create a new binary column: Is_Married
# 'Mrs' title → married woman → 1
# everyone else → 0
# This is another example of feature construction using the split Title
df['Is_Married'] = 0
df.loc[df['Title'] == 'Mrs', 'Is_Married'] = 1

df[['Name', 'Title', 'Is_Married']].head(10)

,Name,Title,Is_Married
0,"Braund, Mr. Owen Harris",Mr,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,1
2,"Heikkinen, Miss. Laina",Miss,0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,1
4,"Allen, Mr. William Henry",Mr,0
5,"Moran, Mr. James",Mr,0
6,"McCarthy, Mr. Timothy J",Mr,0
7,"Palsson, Master. Gosta Leonard",Master,0
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",Mrs,1
9,"Nasser, Mrs. Nicholas (Adele Achem)",Mrs,1


## Summary

### Feature Construction:
```
SibSp + Parch + 1  →  Family_size  →  Family_type (0=alone, 1=small, 2=large)
```
Created a more meaningful feature from raw columns — improved model accuracy.

### Feature Splitting:
```
'Braund, Mr. Owen Harris'  →  Title: 'Mr'
```
Extracted hidden information from a raw text column.

### Key takeaway:
Raw data is often not in the best form for a model.
Feature engineering (construction + splitting) helps the model **understand the data better**
and almost always improves accuracy.